# Lightweight HBCC — PlantVillage Training Pipeline

Trains all models on the PlantVillage plant disease dataset (38 classes, ~70 K training images).
Images are resized to 32×32 to match the HBCC CIFAR configuration exactly.

**Required Kaggle dataset:** `vipoooool/new-plant-diseases-dataset`  
Add it to this notebook before running.

**Three sessions required** (PlantVillage is ~1.6× larger than CIFAR-10 per epoch):
- **Session A** — Teacher (ResNet-18) + lightweight baselines (MobileNetV2, ShuffleNetV2, CoC)
- **Session B** — HBCC-Small and HBCC-Medium CE-only training
- **Session C** — ResNet-18 teacher (re-train) + HBCC KD training + all benchmarks + tables

Enable only the phases for the current session before running all cells.

In [ ]:
!git clone https://github.com/nvhungus/Lightweight-Context-Cluster.git
%cd Lightweight-Context-Cluster

In [ ]:
from __future__ import annotations

import json
import subprocess
import sys
import time
from pathlib import Path

import pandas as pd
from IPython.display import Markdown, display

ROOT = Path.cwd()
if not (ROOT / "tools" / "train.py").exists():
    ROOT = ROOT.parent
assert (ROOT / "tools" / "train.py").exists(), f"Repo root not found. Got {ROOT}"

PYTHON = Path(sys.executable)
print("Repo:", ROOT)
print("Python:", PYTHON)

## Phase Switches

**Session A:** set `RUN_TEACHER`, `RUN_LIGHTWEIGHT_BASELINES`, `RUN_COC_BASELINE` = True; all others False.  
**Session B:** set `RUN_STUDENT_CE` = True; all others False.  
**Session C:** set `RUN_TEACHER_KD`, `RUN_KD`, `RUN_BENCHMARKS` = True; all others False.

In [ ]:
# ── Session A ──────────────────────────────────────────────────────────────
RUN_ENV_CHECKS          = True
RUN_TEACHER             = True   # ResNet-18 reference baseline
RUN_LIGHTWEIGHT_BASELINES = True  # MobileNetV2, ShuffleNetV2
RUN_COC_BASELINE        = True   # CoC pure-cluster baseline

# ── Session B ──────────────────────────────────────────────────────────────
RUN_STUDENT_CE          = False  # HBCC-Small and HBCC-Medium CE-only

# ── Session C ──────────────────────────────────────────────────────────────
RUN_TEACHER_KD          = False  # Re-train teacher for KD (same dataset, different dir)
RUN_KD                  = False  # HBCC-Small and HBCC-Medium with KD
RUN_BENCHMARKS          = False  # Latency / memory / MACs for all models

# ── Epoch budgets ──────────────────────────────────────────────────────────
FULL_EPOCHS     = 150   # ResNet-18, MobileNetV2, ShuffleNetV2, CoC
ACCURACY_EPOCHS = 200   # HBCC CE and KD variants

# ── PlantVillage dataset paths ─────────────────────────────────────────────
# Adjust if the Kaggle dataset is mounted at a different location.
_PV_ROOT = "/kaggle/input/new-plant-diseases-dataset/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)"
PLANT_TRAIN = f"{_PV_ROOT}/train"
PLANT_VALID = f"{_PV_ROOT}/valid"

PLANTVILLAGE_BASE = [
    "data.name=plantvillage",
    f"data.train_dir={PLANT_TRAIN}",
    f"data.val_dir={PLANT_VALID}",
    "model.num_classes=38",
]

# ── Benchmark settings ─────────────────────────────────────────────────────
BENCHMARK_OUTPUT  = "results/benchmark_plantvillage"
BENCHMARK_BATCHES = [1, 16, 64, 128]
BENCHMARK_WARMUP  = 30
BENCHMARK_RUNS    = 100

DEBUG_BENCHMARK_BATCHES = [1, 16]
DEBUG_BENCHMARK_WARMUP  = 2
DEBUG_BENCHMARK_RUNS    = 3

In [ ]:
def _looks_like_tqdm(line: str) -> bool:
    text = line.strip()
    return "%|" in text or text.startswith("eval:") or text.startswith("train ")


def run_py(args: list[str], check: bool = True) -> int:
    cmd = [str(PYTHON), *args]
    print("\n$", " ".join(cmd))
    start = time.perf_counter()
    process = subprocess.Popen(
        cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    assert process.stdout is not None
    progress_handle = display(Markdown(""), display_id=True)
    last_progress = ""
    for line in process.stdout:
        clean = line.rstrip("\r\n")
        if _looks_like_tqdm(clean):
            last_progress = clean
            progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
        else:
            print(line, end="")
    code = process.wait()
    elapsed = time.perf_counter() - start
    if last_progress:
        progress_handle.update(Markdown(f"```text\n{last_progress}\n```"))
    print(f"\n[exit={code}] elapsed={elapsed:.1f}s")
    if check and code != 0:
        raise RuntimeError(f"Command failed with exit code {code}: {' '.join(cmd)}")
    return code


def override_args(overrides: list[str] | None) -> list[str]:
    args: list[str] = []
    for item in overrides or []:
        args.extend(["--override", item])
    return args


def pv_overrides(experiment_name: str | None = None, overrides: list[str] | None = None) -> list[str]:
    items = [*PLANTVILLAGE_BASE]
    if experiment_name is not None:
        items.append(f"experiment.name={experiment_name}")
    items.extend(overrides or [])
    return items


def train(
    config: str,
    output: str,
    experiment_name: str,
    overrides: list[str] | None = None,
    extra: list[str] | None = None,
) -> None:
    args = ["tools/train.py", "--config", config, "--output", output]
    args += override_args(pv_overrides(experiment_name, overrides))
    args += extra or []
    run_py(args)


def benchmark(
    config: str,
    experiment_name: str,
    checkpoint: str | None = None,
    debug: bool = False,
    profile: bool = True,
) -> None:
    batches = DEBUG_BENCHMARK_BATCHES if debug else BENCHMARK_BATCHES
    warmup  = DEBUG_BENCHMARK_WARMUP  if debug else BENCHMARK_WARMUP
    runs    = DEBUG_BENCHMARK_RUNS    if debug else BENCHMARK_RUNS
    args = [
        "tools/benchmark.py", "--config", config,
        "--output", BENCHMARK_OUTPUT,
        "--batch-sizes", *[str(v) for v in batches],
        "--warmup", str(warmup),
        "--runs", str(runs),
    ]
    args += override_args(pv_overrides(experiment_name))
    if checkpoint and Path(ROOT / checkpoint).exists():
        args += ["--checkpoint", checkpoint]
    elif checkpoint:
        print(f"Skip checkpoint for {config}; not found: {checkpoint}")
    if profile:
        args.append("--profile")
    run_py(args)

## 0. Environment Checks

In [ ]:
if RUN_ENV_CHECKS:
    run_py(["-m", "pytest"])
    run_py([
        "tools/shape_trace.py", "--config", "configs/hbcc_accuracy_medium.yaml",
        "--batch-size", "1",
        *override_args(["model.num_classes=38"]),
    ])

## Session A — Teacher and Lightweight Baselines

All baselines trained for `FULL_EPOCHS` (150) with full augmentation.  
Estimated time: ~10 h on a Kaggle T4 GPU.

In [ ]:
EXP_RESNET    = "resnet18_pv"
DIR_TEACHER   = "runs_pv_teacher"
TEACHER_CHECKPOINT = f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"

if RUN_TEACHER:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER,
        EXP_RESNET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_MOBILENET  = "mobilenet_v2_pv"
EXP_SHUFFLENET = "shufflenet_v2_x1_0_pv"
DIR_BASELINES  = "runs_pv_baselines"

if RUN_LIGHTWEIGHT_BASELINES:
    train(
        "configs/baselines/mobilenet_v2_cifar.yaml",
        DIR_BASELINES,
        EXP_MOBILENET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )
    train(
        "configs/baselines/shufflenet_v2_x1_0_cifar.yaml",
        DIR_BASELINES,
        EXP_SHUFFLENET,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_COC    = "coc_baseline_pv"
DIR_COC    = "runs_pv_coc"
COC_CHECKPOINT = f"{DIR_COC}/{EXP_COC}/best.pth"

if RUN_COC_BASELINE:
    train(
        "configs/coc_cifar_baseline.yaml",
        DIR_COC,
        EXP_COC,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

## Session B — HBCC CE-only Training

Both HBCC variants trained with cross-entropy only (no KD).  
Estimated time: ~8 h on a Kaggle T4 GPU.

In [ ]:
EXP_HBCC_SML = "hbcc_accuracy_small_pv"
EXP_HBCC_MED = "hbcc_accuracy_medium_pv"
DIR_ACCURACY = "runs_pv_accuracy"

if RUN_STUDENT_CE:
    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_ACCURACY,
        EXP_HBCC_SML,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_ACCURACY,
        EXP_HBCC_MED,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}"],
    )

## Session C — KD Training, Benchmarks, and Tables

Re-trains ResNet-18 as the KD teacher (saved to a separate directory so Session A's
checkpoint is not overwritten), then distils into both HBCC variants.  
Estimated time: ~11 h on a Kaggle T4 GPU.

> **Tip:** If you saved Session A's output as a Kaggle dataset and added it as input,
> set `RUN_TEACHER_KD = False` and update `TEACHER_KD_CHECKPOINT` to point to that
> pre-trained checkpoint instead.

In [ ]:
EXP_RESNET_KD       = "resnet18_pv_kd"
DIR_TEACHER_KD      = "runs_pv_teacher_kd"
TEACHER_KD_CHECKPOINT = f"{DIR_TEACHER_KD}/{EXP_RESNET_KD}/best.pth"

# If Session A's teacher is available as a dataset input, override this path:
# TEACHER_KD_CHECKPOINT = "/kaggle/input/<session-a-dataset>/runs_pv_teacher/resnet18_pv/best.pth"

if RUN_TEACHER_KD:
    train(
        "configs/baselines/resnet18_cifar.yaml",
        DIR_TEACHER_KD,
        EXP_RESNET_KD,
        overrides=[f"train.epochs={FULL_EPOCHS}"],
    )

In [ ]:
EXP_HBCC_SML_KD = "hbcc_accuracy_small_kd_pv"
EXP_HBCC_MED_KD = "hbcc_accuracy_medium_kd_pv"
DIR_KD          = "runs_pv_kd"

KD_OVERRIDES = ["train.kd_alpha=0.5", "train.kd_temperature=4.0"]

if RUN_KD:
    teacher_path = ROOT / TEACHER_KD_CHECKPOINT
    if not teacher_path.exists():
        raise FileNotFoundError(
            f"Teacher checkpoint not found: {teacher_path}\n"
            "Run Session A (RUN_TEACHER=True) or set RUN_TEACHER_KD=True first."
        )
    print(f"Teacher checkpoint exists: True → {teacher_path}")

    train(
        "configs/hbcc_accuracy_small.yaml",
        DIR_KD,
        EXP_HBCC_SML_KD,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )
    train(
        "configs/hbcc_accuracy_medium.yaml",
        DIR_KD,
        EXP_HBCC_MED_KD,
        overrides=[f"train.epochs={ACCURACY_EPOCHS}", *KD_OVERRIDES],
        extra=[
            "--teacher-config", "configs/baselines/resnet18_cifar.yaml",
            "--teacher-checkpoint", TEACHER_KD_CHECKPOINT,
        ],
    )

## Benchmark Matrix

In [ ]:
BENCHMARK_JOBS = [
    ("configs/baselines/resnet18_cifar.yaml",           EXP_RESNET,       f"{DIR_TEACHER}/{EXP_RESNET}/best.pth"),
    ("configs/baselines/mobilenet_v2_cifar.yaml",       EXP_MOBILENET,    f"{DIR_BASELINES}/{EXP_MOBILENET}/best.pth"),
    ("configs/baselines/shufflenet_v2_x1_0_cifar.yaml", EXP_SHUFFLENET,   f"{DIR_BASELINES}/{EXP_SHUFFLENET}/best.pth"),
    ("configs/coc_cifar_baseline.yaml",                 EXP_COC,          COC_CHECKPOINT),
    ("configs/hbcc_accuracy_small.yaml",                EXP_HBCC_SML,     f"{DIR_ACCURACY}/{EXP_HBCC_SML}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",               EXP_HBCC_MED,     f"{DIR_ACCURACY}/{EXP_HBCC_MED}/best.pth"),
    ("configs/hbcc_accuracy_small.yaml",                EXP_HBCC_SML_KD,  f"{DIR_KD}/{EXP_HBCC_SML_KD}/best.pth"),
    ("configs/hbcc_accuracy_medium.yaml",               EXP_HBCC_MED_KD,  f"{DIR_KD}/{EXP_HBCC_MED_KD}/best.pth"),
]

if RUN_BENCHMARKS:
    for cfg, exp_name, ckpt in BENCHMARK_JOBS:
        if not (ROOT / cfg).exists():
            print("Skip missing config:", cfg)
            continue
        benchmark(cfg, exp_name, checkpoint=ckpt)

## Results Tables

In [ ]:
PV_RUN_DIRS = [
    DIR_TEACHER, DIR_BASELINES, DIR_COC, DIR_ACCURACY,
    DIR_TEACHER_KD, DIR_KD,
]


def read_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    rows = []
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.strip():
            rows.append(json.loads(line))
    return rows


def collect_train_metrics() -> pd.DataFrame:
    rows = []
    for run_root in PV_RUN_DIRS:
        for metrics_path in (ROOT / run_root).glob("*/metrics.jsonl"):
            records = read_jsonl(metrics_path)
            val_records = [r for r in records if "val_acc1" in r]
            if not val_records:
                continue
            best = max(val_records, key=lambda r: r["val_acc1"])
            test_records  = [r for r in records if "test_acc1" in r]
            test_record   = test_records[-1] if test_records else {}
            train_records = [r for r in records if "train_acc1" in r]
            last_train    = train_records[-1] if train_records else {}
            rows.append({
                "run_dir":        str(metrics_path.parent.relative_to(ROOT)),
                "experiment":     metrics_path.parent.name,
                "best_epoch":     best.get("epoch"),
                "best_val_acc1":  best.get("val_acc1"),
                "best_val_acc5":  best.get("val_acc5"),
                "test_acc1":      test_record.get("test_acc1"),
                "test_acc5":      test_record.get("test_acc5"),
                "last_train_acc1": last_train.get("train_acc1"),
            })
    return pd.DataFrame(rows).sort_values("best_val_acc1", ascending=False) if rows else pd.DataFrame()


train_df = collect_train_metrics()
train_df

In [ ]:
def collect_benchmarks() -> pd.DataFrame:
    rows = []
    for path in (ROOT / BENCHMARK_OUTPUT).glob("*.json"):
        rec = json.loads(path.read_text(encoding="utf-8"))
        rows.append(rec)
    return pd.DataFrame(rows) if rows else pd.DataFrame()


bench_df = collect_benchmarks()
cols = ["config_id", "model_name", "params_total", "macs", "latency_ms_b1", "peak_memory_mb"]
bench_df[[c for c in cols if c in bench_df.columns]] if not bench_df.empty else bench_df

In [ ]:
def build_publication_table(bench: pd.DataFrame, train: pd.DataFrame) -> pd.DataFrame:
    if bench.empty:
        return bench
    out = bench.copy()
    out["acc1"] = None
    out["acc5"] = None
    if not train.empty:
        acc_source = "test_acc1" if "test_acc1" in train.columns else "best_val_acc1"
        acc1_map = dict(zip(train["experiment"], train[acc_source].fillna(train["best_val_acc1"])))
        acc5_map = {}
        if "test_acc5" in train.columns:
            acc5_map = dict(zip(train["experiment"], train["test_acc5"].fillna(train.get("best_val_acc5", 0))))
        for idx, row in out.iterrows():
            name = row.get("config_id")
            out.at[idx, "acc1"] = acc1_map.get(name)
            out.at[idx, "acc5"] = acc5_map.get(name)
    rename = {
        "config_id":       "Model",
        "acc1":            "Top-1 Acc (%)",
        "acc5":            "Top-5 Acc (%)",
        "params_total":    "Params",
        "macs":            "MACs",
        "latency_ms_b1":   "Latency b1 (ms)",
        "peak_memory_mb":  "Peak Memory (MB)",
    }
    table = out[[c for c in rename if c in out.columns]].rename(columns=rename).copy()
    if "Top-1 Acc (%)" in table.columns:
        table = table.sort_values("Top-1 Acc (%)", ascending=False)
    return table.reset_index(drop=True)


pub_df = build_publication_table(bench_df, train_df)
out_dir = ROOT / "results"
out_dir.mkdir(exist_ok=True)
pub_df.to_csv(out_dir / "plantvillage_published_results.csv", index=False)
pub_df